In [1]:
#Extração dos dados(E)
import pandas as pd
import numpy as np
import os 
from sqlalchemy import create_engine

def extracao_dados():
    pasta_datasets = 'datasets/brutos'
    tabelas_origem = {}

    print("==========INICIANDO EXTRAÇÃO DOS DADOS==========".center(30))

    for arquivo in os.listdir(pasta_datasets):

        if arquivo.endswith('.csv'):

            caminho = os.path.join(pasta_datasets,arquivo)
            nome_chave = arquivo.replace('.csv','')
            print(f"Lendo e carregando {arquivo}")
            tabelas_origem[nome_chave] = pd.read_csv(caminho)

    print(f'Extração concluida!!! Total de arquivos na memória: {len(tabelas_origem)}')
    return tabelas_origem


tabelas_brutas = extracao_dados()

==========INICIANDO EXTRAÇÃO DOS DADOS==========
Lendo e carregando olist_customers_dataset.csv
Lendo e carregando olist_orders_dataset.csv
Lendo e carregando olist_order_items_dataset.csv
Lendo e carregando olist_order_payments_dataset.csv
Lendo e carregando olist_order_reviews_dataset.csv
Lendo e carregando olist_products_dataset.csv
Lendo e carregando olist_sellers_dataset.csv
Extração concluida!!! Total de arquivos na memória: 7


In [2]:
#Transformação dos dados(T)- Tratamento por tabela

df_produtos = tabelas_brutas['olist_products_dataset']
df_pedidos = tabelas_brutas['olist_orders_dataset']
df_clientes = tabelas_brutas['olist_customers_dataset']
df_vendedores = tabelas_brutas['olist_sellers_dataset']
df_reviews = tabelas_brutas['olist_order_reviews_dataset']
df_itens_pedido = tabelas_brutas['olist_order_items_dataset']
df_itens_pagamento = tabelas_brutas['olist_order_payments_dataset']

def trat_itens_pag(df_itens_pagamento):
    df_itens_pagamento["payment_type"] = df_itens_pagamento["payment_type"].astype(str).str.title()
    df_itens_pagamento["payment_type"] = df_itens_pagamento["payment_type"].str.replace('_',' ')
    df_itens_pagamento["payment_sequential"] = df_itens_pagamento["payment_sequential"].astype(int)
    df_itens_pagamento["payment_installments"] = df_itens_pagamento["payment_installments"].astype(int)
    return df_itens_pagamento


def trat_itens_pedido(df_itens_pedido):
    df_itens_pedido["shipping_limit_date"]= pd.to_datetime(df_itens_pedido["shipping_limit_date"])
    return df_itens_pedido


def trat_reviews(df_reviews):
    """Faz o tratamento da tabela de fatos de reviews(removendo comentarios e colunas nao uteis para analise)"""
    df_reviews = df_reviews[["review_id","order_id","review_score","review_creation_date","review_answer_timestamp"]]
    df_reviews["review_creation_date"] = pd.to_datetime(df_reviews["review_creation_date"])
    df_reviews["review_answer_timestamp"] = pd.to_datetime(df_reviews["review_answer_timestamp"])
    df_reviews["order_id"] = df_reviews["order_id"].astype(str)
    df_reviews["review_id"] = df_reviews["review_id"].astype(str)
    df_reviews["review_score"] = df_reviews["review_score"].astype(int)
    return df_reviews
    

def trat_vendedores(df_vendedores):
    """Faz o tratamento da tabela de registro de vendedores"""
    df_vendedores = df_vendedores.drop_duplicates()
    df_vendedores["seller_city"] = df_vendedores["seller_city"].astype(str)
    df_vendedores["seller_city"] = df_vendedores["seller_city"].str.title()
    df_vendedores["seller_state"] = df_vendedores["seller_state"].str.strip()
    return df_vendedores


def trat_clientes(df_clientes):
    """Faz o tratamento dos dados da tabela de registro de clientes"""
    df_clientes = df_clientes.drop_duplicates()
    df_clientes["customer_city"] = df_clientes["customer_city"].astype(str).str.title()
    df_clientes["customer_zip_code_prefix"] = df_clientes["customer_zip_code_prefix"].astype(int)
    return df_clientes



def trat_pedidos(df_pedidos):
    """Faz o tratamento dos dados da tabela de registro de pedidos"""
    #Corrigindo coluna de data:
    colunas_data = ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
                    "order_delivered_customer_date","order_estimated_delivery_date"]
    for coluna in colunas_data:
        df_pedidos[coluna] = pd.to_datetime(df_pedidos[coluna],errors='coerce').dt.normalize()

    df_pedidos["order_status"] = df_pedidos["order_status"].str.strip()

    return df_pedidos 


def trat_produtos(df_produtos):
    """Faz o tratamento dos dados da tabela de registro de produtos"""
    df_produtos = df_produtos.drop_duplicates()
    df_produtos["product_category_name"] = df_produtos["product_category_name"].astype(str).str.replace("_",' ')
    df_produtos["product_category_name"] = df_produtos["product_category_name"].str.strip()
    df_produtos["product_category_name"] = df_produtos["product_category_name"].str.title()

    lista_colunas = ["product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g",
                                "product_length_cm","product_height_cm","product_width_cm"]
    
    df_produtos[lista_colunas] = df_produtos[lista_colunas].fillna(0)
    return df_produtos

# Função principal de transformação dos dados:

def tratamento_dados(tabelas_brutas):

    tabelas_tratadas = {}

    try:
        print("==========INICIANDO TRANSFORMAÇÃO DOS DADOS==========".center(30))
        tabelas_tratadas["d_produtos"]= trat_produtos(tabelas_brutas['olist_products_dataset'])
        tabelas_tratadas["d_clientes"] = trat_clientes(tabelas_brutas['olist_customers_dataset'])
        tabelas_tratadas["d_vendedores"] = trat_vendedores(tabelas_brutas['olist_sellers_dataset'])
        tabelas_tratadas["f_pedidos"] = trat_pedidos(tabelas_brutas['olist_orders_dataset'])
        tabelas_tratadas["f_pagamentos"] = trat_itens_pag(tabelas_brutas['olist_order_payments_dataset'])
        tabelas_tratadas["f_reviews"] = trat_reviews(tabelas_brutas['olist_order_reviews_dataset'])
        tabelas_tratadas["f_itens"] = trat_itens_pedido(tabelas_brutas['olist_order_items_dataset'])
        print("Transformação concluída!!!")
    except Exception as e:
        print(f"Erro ao tratar os dados, detalhes: {e}")
        
    return tabelas_tratadas
    
tabelas_processadas = tratamento_dados(tabelas_brutas)   

==========INICIANDO TRANSFORMAÇÃO DOS DADOS==========
Transformação concluída!!!


C:\Users\ys100\AppData\Local\Temp\ipykernel_9712\2836932297.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews["review_creation_date"] = pd.to_datetime(df_reviews["review_creation_date"])
C:\Users\ys100\AppData\Local\Temp\ipykernel_9712\2836932297.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews["review_answer_timestamp"] = pd.to_datetime(df_reviews["review_answer_timestamp"])
C:\Users\ys100\AppData\Local\Temp\ipykernel_9712\2836932297.py:29: SettingWithCopyWarning: 
A value is t

In [21]:
#Carregamento de dados(L)

from dotenv import load_dotenv

load_dotenv(override=True)


def carregamento_dados(tabelas_processadas):
    
    host = os.getenv("host")
    database = os.getenv("database")
    user = os.getenv("usuario")
    port = os.getenv("port")
    password = os.getenv("senha_banco")
    url_conexao = f"postgresql://{user}:{password}@{host}:{port}/{database}"

    engine = create_engine(url_conexao)
    print(f"Tentando conectar-se à base de dados: {database}...")
    try:
        with engine.connect() as conn:
            print('Conexão bem sucedida!\n')
            try:
                print("==========CARREGANDO DADOS NO BANCO==========".center(30))

                for nome_tabela,df_tabela in tabelas_processadas.items():
                    print(f"Levando {nome_tabela} para o banco de dados")
                    df_tabela.to_sql(name = nome_tabela,con=engine,if_exists='replace',index=False)
                print("CARREGAMENTO CONCLUÍDO COM SUCESSO!!!")

            except Exception as e:
                return print(f"Não foi possível carregar os dados no banco: {e}")
    
    except Exception as ex:
        return print(f"Erro ao tentar conectar-se ao banco: {ex}")

    
carregamento_dados(tabelas_processadas)


Tentando conectar-se à base de dados: olist_db...
Conexão bem sucedida!

==========CARREGANDO DADOS NO BANCO==========
Levando d_produtos para o banco de dados
Levando d_clientes para o banco de dados
Levando d_vendedores para o banco de dados
Levando f_pedidos para o banco de dados
Levando f_pagamentos para o banco de dados
Levando f_reviews para o banco de dados
Levando f_itens para o banco de dados
CARREGAMENTO CONCLUÍDO COM SUCESSO!!!
